# 1. set-up environments

In [1]:
import json
import os

keys: dict = {}

with open('api-key.json') as key_file:
    keys = json.load(key_file)

os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_API_KEY'] = keys['langchains-personal-key'] 
os.environ['OPENAI_API_KEY'] = keys['openai_note-key']

# 2. set-up sqlite cache.db

In [2]:
from langchain.globals import set_llm_cache
from langchain.cache import SQLiteCache

set_llm_cache(SQLiteCache("cache.db"))

# 3. set-up openai model

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

model = ChatOpenAI(
    model='gpt-3.5-turbo-16k',
    temperature=0.1,
)
str_output_parser = StrOutputParser()

# 4. set-up few shot learning prompts & chains

In [4]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

movie_few_shot_samples = [
    {
        "movie_name": "다크 나이트",
        "ai_answer": """
해당 영화에 대한 알고 있는 것을 소개드립니다!
- 영화 제목: 다크 나이트
- 감독: 크리스토퍼 놀란
- 주요 출연진: 크리스찬 베일, 히스 레저, 아론 에크하트
- 개봉일: 2008.08.06.
- 러닝 타임: 152분
- 장르: 액션, 범죄, 드라마, 미스터리 
- 국가: 미국
- 예산: $185,000,000 (추정)
- 흥행 수익: $1,008,524,647
- 시놉시스: 정의로운 지방 검사 ‘하비 덴트’, ‘짐 고든’ 반장과 함께 범죄 소탕 작전을 펼치며 범죄와 부패로 들끓는 고담시를 지켜나가는 ‘배트맨’ 그러던 어느 날, 살아남기 위해 발버둥치던 범죄 조직은 배트맨을 제거하기 위해 광기어린 악당 ‘조커’를 끌어들이고 정체를 알 수 없는 조커의 등장에 고담시 전체가 깊은 혼돈 속으로 빠져든다. 급기야 배트맨을 향한 강한 집착을 드러낸 조커는 그가 시민들 앞에 정체를 밝힐 때까지 매일 새로운 사람들을 죽이겠다 선포하고 배트맨은 사상 최악의 악당 조커를 막기 위해 자신의 모든 것을 내던진 마지막 대결을 준비한다.
        """
    },
    {
        "movie_name": "나는 내일 어제의 너와 만난다",
        "ai_answer": """
해당 영화에 대한 알고 있는 것을 소개드립니다!
- 영화 제목: 나는 내일 어제의 너와 만난다
- 감독: 미키 타카히로
- 주요 출연진: 후쿠시 소우타, 고마츠 나나
- 개봉일: 2017.10.12.
- 러닝 타임: 111분
- 장르: 멜로, 로맨스, 판타지
- 국가: 일본
- 예산: $3,000,000 (추정)
- 흥행 수익: $14,483,358
- 시놉시스: 스무 살의 ‘타카토시’는 지하철에서 우연히 만난 ‘에미’를 보고 순식간에 마음을 빼앗긴다. 운명 같은 끌림을 느낀 타카토시의 고백으로 두 사람은 연인이 되고, 매일 만나 행복한 데이트를 한다. 하지만, 왠지 종종 의미를 알 수 없는 눈물을 보이던 에미로부터 믿을 수 없는 비밀을 듣게 된 타카토시는 큰 혼란에 빠진다. 그 비밀은 바로 타카토시와 에미의 시간은 서로 반대로 흐르고 있고, 교차되는 시간 속에서 함께 할 수 있는 시간은 오직 30일뿐이라는 것. 30일 후에도, 이 사랑은 계속될 수 있을까?
        """
    },
]

movie_prompt = ChatPromptTemplate.from_messages([
    ("human", "영화 {movie_name} 에 대해서 알고있는 것 말해줘."),
    ("ai", "{ai_answer}"),
])
few_shot_movie_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=movie_prompt,
    examples=movie_few_shot_samples,
)
chat_movie_prompt = ChatPromptTemplate.from_messages([
    ("system", "넌 모든 영화 데이터를 알고 있는 영화 전문가야. 영화 스펙애 대해서 가볍게 말해줘야해."),
    few_shot_movie_prompt,
    ("human", "영화 {movie_name} 에 대해서 알고있는 것 말해줘."),
])

movie_chain = chat_movie_prompt | model | str_output_parser

# 5. invokes

In [5]:
print(movie_chain.invoke({"movie_name": "포레스트 검프"}))


해당 영화에 대한 알고 있는 것을 소개드립니다!
- 영화 제목: 포레스트 검프
- 감독: 로버트 저메키스
- 주요 출연진: 톰 행크스, 로빈 라이트 펜, 게리 신즈
- 개봉일: 1994.07.06.
- 러닝 타임: 142분
- 장르: 드라마, 로맨스
- 국가: 미국
- 예산: $55,000,000 (추정)
- 흥행 수익: $677,945,399
- 시놉시스: 지능이 낮은 포레스트 검프는 어려운 환경에서도 순수하고 착한 마음을 가진 남자로서, 우연한 사건들을 통해 성공을 거두게 된다. 그는 미국 역사의 여러 사건에 직접 관여하고, 탁월한 탁구 실력으로 선수로서의 명성을 얻으며, 사람들과의 만남을 통해 사랑과 친구를 찾아가는 이야기를 그려낸다. 그러나 그의 가장 큰 사랑은 언제나 그를 지지하고 사랑해주는 제니였고, 그녀와의 사랑은 시간과 공간을 초월하여 영원한 것으로 남는다.
        


In [6]:
# 최근 데이터라 잘 못 나오는 듯
print(movie_chain.invoke({"movie_name": "극장총집편 봇치 더 록! 후편"}))


해당 영화에 대한 알고 있는 것을 소개드립니다!
- 영화 제목: 극장총집편 봇치 더 록! 후편
- 감독: 마이크 미첼
- 주요 출연진: 드웨인 존슨, 케빈 하트, 잭 블랙, 카렌 길런
- 개봉일: 2019.12.13.
- 러닝 타임: 123분
- 장르: 액션, 모험, 코미디, 판타지
- 국가: 미국
- 예산: $125,000,000 (추정)
- 흥행 수익: $796,575,993
- 시놉시스: 이번에도 ‘스펜서’, ‘프릿지’, ‘베서니’는 게임 세계에서 벗어나 현실 세계로 들어오게 된다. 그런데 이번에는 게임 안에서의 모험뿐만 아니라, 현실 세계에서도 위험한 상황에 처하게 된다. 게임 속에서는 다양한 환경과 캐릭터들과의 대결을 펼치며 생존을 위해 모험을 떠나지만, 동시에 게임 밖에서는 현실 세계에서도 위험한 임무를 수행해야 한다. 이들은 게임 속에서의 모험과 현실 세계에서의 위험을 동시에 해결하며, 친구들과 함께 극장총집편의 세계를 구하기 위해 모험을 떠난다.


In [7]:
print(movie_chain.invoke({"movie_name": "인사이드아웃"}))


해당 영화에 대한 알고 있는 것을 소개드립니다!
- 영화 제목: 인사이드 아웃 (Inside Out)
- 감독: 피트 닥터
- 주요 출연진 (성우): 에이미 포엘러, 필리스 스미스, 리처드 커스
- 개봉일: 2015.06.19.
- 러닝 타임: 95분
- 장르: 애니메이션, 코미디, 판타지, 가족
- 국가: 미국
- 예산: $175,000,000 (추정)
- 흥행 수익: $857,611,174
- 시놉시스: 영화는 11살 소녀 ‘라일리’의 머릿속에서 벌어지는 이야기를 그린다. ‘라일리’는 미국 중서부에서 행복하게 살고 있는데, 어느 날 가족의 이사로 인해 새로운 도시로 이사를 가야만 한다. 이사로 인해 ‘라일리’의 감정들은 혼란에 빠지게 되고, 그들은 ‘라일리’의 머릿속에서 펼쳐지는 모험을 통해 서로를 이해하고 성장하게 된다. 영화는 기쁨, 슬픔, 분노, 혐오, 두려움 등 다양한 감정들을 통해 인간의 감정과 성장을 다루고 있다.
